# OpenAI API (Chat Completions)

Source: 
- [list of openai available models](https://developers.openai.com/api/docs/models)
- [developers.openai.com/api/docs/ overview http reference](https://developers.openai.com/api/reference/overview) to understand how to start
- [OpenAI usage dashboard](https://platform.openai.com/settings/organization/usage) to monitor costs

Note: 
- In the following example we'll see Chat completions `/v1/chat/completions` that is the older endpoint, still fully supported, alongside the new Responses API.
- We're using chat completions because as of today (2026/08) this is the one supported by every other providers, as times go on I expect that providers will migrate to [OpenResponses](https://www.openresponses.org/specification)

In real-life code, you won't use httpx to use LLM because SDKs are available.

SDKs gives you for free:
- Idempotency key on each request (if a timeout occurs you don't pay your request twice)
- Http client pooling (they are built on top of httpx.client)
- Retries with exponential backoffs or retry-after on selected errors (429, 5xx)
- Structured output with pydantic
- SSE parsing when using streams with easier error handling (just a custom exception)

What you need to add on top of API:
- cost tracking per run (not per single completion)
- rate limiting to avoid 429
- evals
- tracing to determine where LLM didn't answer as expected  
- cache completions in dev to reduce costs during development
- what to do when an error occurs

Gotchas (hard-learned):
- Reasoning models (gpt-5 family) reject `temperature`, `top_p`, penalties and `logit_bias`.
- Streaming: declare `stream_options: {"include_usage": true}` or you get no usage at all when streaming.

Gotchas (hard-learned but resolved by SDKs):
- Streaming: the usage chunk arrives with an empty `choices` array.
- Streaming: `raise_for_status()` fires before the body is read, if there is an error call `r.read()` first.
- Streaming: errors after the initial 200 travel *inside* the stream within a chunk with an error key instead of the choices key 
- Streaming: a missing `[DONE]` is the only signal of a truncated response.


## Setup

1. Create API Key: https://platform.openai.com/api-keys
2. Insert the API key in .env
    OPENAI_API_KEY=<...>

## Test Raw API (No SDK just httpx)

source: [/v1/chat/completions/ - reference](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create)

### Create a Chat Completion

In [ ]:
import httpx
from dotenv import load_dotenv
import os

# load .env vars to environment variables
load_dotenv() 

# OpenAI
r = httpx.post(
    "https://api.openai.com/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"
    },
    json={
        "model": "gpt-5-nano",
        "messages": [
            { "role": "system", "content": "You are a helpful friend." }, # same as developer, kept for cross-provider portability
            { "role": "user", "content": "Hello" },
        ],
        "reasoning_effort": "minimal"
    },
)

print(f"r: {r}")
print(f"status_code: {r.status_code}")
print(f"text: {r.text}")
print(f"json: {r.json()}\r\n")
print(f"choices[0]>message>content: {r.json()["choices"][0]["message"]["content"]}")

r: <Response [200 OK]>
status_code: 200
text: {
  "id": "chatcmpl-EItsmnv89dgA2CgxqIAlG0mKS6bAd",
  "object": "chat.completion",
  "created": 1788173424,
  "model": "gpt-5-nano-2025-08-07",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "Hi there! How can I help today?",
        "refusal": null,
        "annotations": []
      },
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "prompt_tokens": 17,
    "completion_tokens": 18,
    "total_tokens": 35,
    "prompt_tokens_details": {
      "cached_tokens": 0,
      "audio_tokens": 0
    },
    "completion_tokens_details": {
      "reasoning_tokens": 0,
      "audio_tokens": 0,
      "accepted_prediction_tokens": 0,
      "rejected_prediction_tokens": 0
    }
  },
  "service_tier": "default",
  "system_fingerprint": null
}

json: {'id': 'chatcmpl-EItsmnv89dgA2CgxqIAlG0mKS6bAd', 'object': 'chat.completion', 'created': 1788173424, 'model': 'gpt-5-nano-2025-08-07', 'choices':

#### Request params

- header authorization = openai api key
- body
    - model = the model to use (gpt-4o-mini / gpt-5.6-luna / gpt-5.6-sol). 
        - See costs and specs on the previous link
        - gpt-5.6-luna 0.2$-1.2$/Mtoken - optimized for cost-sensitive ops            
        - gpt-5-nano 0.05$-0.4$/MToken (input-output) - fastest and most cost affordable model
        - gpt-4o-mini 0.15$-0.6$/Mtoken - fast and affordable model
    - messages = list of messaging comprising the conversation so far
    - tools = list of the tools that the model can call 
    - tool_choice = 
        - none: the model cannot call any tool
        - required: the model must call at least a tool
        - auto: the model can decide generate a message or call one or more tools
    - parallel_tool_calls = if false, forces one tool call at a time
    - n = number of chat completions to generate (usually 1, used for alternatives answers)
    - stream = if true, the response will be streamed
    - stream_options = {"include_usage": True} -> without this, streamed responses carry NO usage block. You lose all cost tracking.
    - response_format = 
        -   { "type": "json_schema", "json_schema": {...} } enables structured output 
        -   { "type": "json_object" } ensures the model generates a valid json output (model won't generate json unless specified in user or system prompt)
    - prompt_cache_key = to optimize cache hit rates
    - max_completion_tokens = max number of tokens that can be generated for a completion (including output and reasoning tokens)
    - logprobs = if true, returns the log probability of each output token
    - seed = best-effort reproducibility across identical requests (useful for evals)
    - reasoning_effort = none, minimal, low, medium, high, xhigh, and max (available values differ according to the specific model)
    - temperature = 0/2, 2 is more random, 0 is more focused and deterministic (unsupported in GPT5 - use reasoning_effort)        
    - frequency_penalty = -2.0/2.0 to penalize new tokens based on their frequency (unsupported in GPT5 - use reasoning_effort)
    - presence_penalty = -2.0/2.0 to penalize new tokens that already appear in the text (unsupported in GPT5 - use reasoning_effort)
    - logit_bias = map[number] (unsupported in GPT5 - use reasoning_effort)
        - introduces a bias for specified tokens -100 (min) is a ban +100 (max) is exclusive choiche -1 is penality +1 is boost and so on

#### Response Params

- id = unique identifier for the chat complation (useful to store or retrieve a previous chat completion that had store parameter set to true)
- created = unix timestamp
- choices[0] = list of chat completion choices (usually is one but can be more than one if `n` is greater than 1)
    - .message.content = generated message (None if the model calls a tool)
    - .message.tool_calls = present only on tool calls (None if the model generated a message)
    - .finish_reason = why generation stopped:
        - "stop"           -> completed normally
        - "length"         -> hit max_tokens, output is TRUNCATED
        - "tool_calls"     -> model wants to call a tool
        - "content_filter" -> blocked
- usage
    - .prompt_tokens / completion_tokens / total_tokens = billing
    - .completion_tokens_details.reasoning_tokens = reasoning tokens, billed as output but never returned (gpt-5 family)
    - .prompt_tokens_details.cached_tokens = input tokens billed at reduced rate

### Stream Chat Completions

Use a stream to retrieve chunks returned from the model using SSE

In [34]:
import json
import httpx
from dotenv import load_dotenv
import os

# load .env vars to environment variables
load_dotenv() 

# here we use client.stream, if we used client.post the stream would be buffered by httpx and couldn't see them
# if the stream drops or read is reached an exception will be raised by httpx
with httpx.Client(timeout=httpx.Timeout(connect=5.0, read=60.0, write=10.0, pool=5.0)) as client:
    with client.stream(
        "POST",
        "https://api.openai.com/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"
        },
        json= {
            "model": "gpt-5-nano",
            "messages": [
                {"role": "system", "content": "You are a helpful friend."},
                {"role": "user", "content": "List three kinds of label adhesives."},
            ],
            "reasoning_effort": "minimal",
            "stream": True,
            "stream_options": {"include_usage": True},  # without this you won't receive the final usage field
        },
    ) as r:
        if r.is_error:
            r.read()
            try:
                err = r.json()["error"]
                msg = f"{r.status_code} {err.get('code') or err['type']}: {err['message']}"
            except Exception: # needed to handle errors not in OpenAI format (e.g. a proxy error)
                msg = f"{r.status_code}: {r.text[:500]}"
            raise RuntimeError(f"{msg} (request-id: {r.headers.get('x-request-id')})") # x-request-id is needed if you need to open a ticket to OpenAI

        lines = [] # keeping them just to print at the end
        pieces = []
        usage = None
        done_received = False
        finish_reason = None

        for line in r.iter_lines():
            # In SSE an event is composed of one or more field lines (data:, event:, id:, retry:,:) followed by an empty line that signals the end of the event 
            # OpenAI sends a single "data: " per event followed by the empty line
            # Note that ":" alone is just a keep alive
            lines.append(line or "<empty line>")
            if not line.startswith("data: "):  
                continue

            data = line[6:]
            if data == "[DONE]": # this could be a break but i keep continue to receive the last empty line
                done_received = True
                continue

            chunk = json.loads(data)

            if chunk.get("error"):
                raise RuntimeError(f"stream error: {chunk['error']}")

            if chunk.get("usage"):
                usage = chunk["usage"]

            if not chunk.get("choices"): # usage chunks usually contains empty choiches, some providers send messages with empty choices at the start
                continue

            choice = chunk["choices"][0]
            delta = choice["delta"].get("content")
            if delta:
                pieces.append(delta)
                print(f"${delta}", end="", flush=True)

            if choice.get("finish_reason"):
                finish_reason = choice["finish_reason"]

        if not done_received or finish_reason is None: # the stream has been interrupted before finish
            raise RuntimeError(f"stream truncated: done={done_received} finish_reason={finish_reason}")

print(f"\n\nfinish_reason: {finish_reason}")
print(f"usage: {usage}")
print(f"pieces: {pieces}")
print(f"\n\nlines: \n{"\n".join(lines)}")

$Here$ are$ three$ common$ types$ of$ label$ adhesives$:

$1$.$ Permanent$ adhesive$:$ Bonds$ strongly$ and$ is$ designed$ to$ stay$ attached$ for$ the$ life$ of$ the$ label$,$ often$ resistant$ to$ water$,$ solvents$,$ and$ temperature$ changes$.
$2$.$ Rem$ovable$ adhesive$:$ Allows$ labels$ to$ be$ peeled$ off$ with$ minimal$ residue$,$ suitable$ for$ temporary$ labeling$ or$ applications$ where$ re$-l$ab$elling$ is$ expected$.
$3$.$ Tack$ vinyl$ or$ solvent$-based$ adhesive$:$ A$ higher$-t$ack$ option$ used$ for$ difficult$ surfaces$ or$ outdoor$ use$,$ providing$ strong$ initial$ grab$ and$ durability$ in$ challenging$ conditions$.

finish_reason: stop
usage: {'prompt_tokens': 23, 'completion_tokens': 111, 'total_tokens': 134, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}
pieces: ['Here', ' are', ' three', ' common', ' types'